In [1]:
import heapq
import time
import numpy as np
from typing import List, Dict
from dataclasses import dataclass
from collections import defaultdict
import random

# --- Data Structures ---

@dataclass
class TaskConfig:
    task_id: int
    arrival_time: float
    expected_duration: float
    deadline: float
    is_io_bound: bool = False
    is_foreground: bool = False  # OS decides boost based on this


@dataclass
class TaskState:
    base_priority: int = 0
    dynamic_priority: int = 0
    start_time: float = -1
    end_time: float = -1
    remaining: float = 0
    executed_time: float = 0
    waiting_time: float = 0

    def __post_init__(self):
        self.dynamic_priority = self.base_priority


class Task:
    def __init__(self, config: TaskConfig):
        self.config = config
        self.state = TaskState(remaining=config.expected_duration)
        self.assign_base_priority()

    def assign_base_priority(self):
        """
        Simulate OS assigning priority based on characteristics
        """
        priority = 8  # default: NORMAL
        if self.config.deadline < 10:
            priority += 4  # tight deadline → higher priority
        if self.config.is_io_bound:
            priority += 2
        if self.config.is_foreground:
            priority += 2
        self.state.base_priority = min(priority, 15)
        self.state.dynamic_priority = self.state.base_priority

    def __lt__(self, other):
        return self.state.dynamic_priority > other.state.dynamic_priority


# --- Metrics ---

class SchedulerMetrics:
    def __init__(self):
        self.task_metrics: Dict[int, Dict] = defaultdict(dict)
        self.system_metrics = {
            "total_context_switches": 0,
            "cpu_utilization": 0,
            "missed_deadlines": 0,
            "jains_fairness": 0,
            "avg_waiting_time": 0,
            "avg_response_time": 0
        }

    def calculate_jains_fairness(self, execution_times: List[float]):
        numerator = sum(execution_times) ** 2
        denominator = len(execution_times) * sum(t ** 2 for t in execution_times)
        return numerator / denominator if denominator != 0 else 0


# --- Scheduler ---

QUANTUM_BASE = 2

class Scheduler:
    def __init__(self):
        self.clock = 0
        self.ready_queue = []
        self.current_task = None
        self.metrics = SchedulerMetrics()
        self.pending_tasks = []
        self.total_busy_time = 0

    def add_task(self, task: Task):
        heapq.heappush(self.pending_tasks, (task.config.arrival_time, task))

    def _update_priorities(self):
        if self.current_task:
            if self.current_task.config.is_io_bound and random.random() < 0.6:
                self.current_task.state.dynamic_priority = min(
                    self.current_task.state.dynamic_priority + 2, 15)
            elif self.clock % 4 == 0:
                self.current_task.state.dynamic_priority = max(
                    self.current_task.state.base_priority,
                    self.current_task.state.dynamic_priority - 1
                )

    def _handle_preemption(self):
        if self.current_task and self.ready_queue:
            next_task = self.ready_queue[0]
            if next_task.state.dynamic_priority > self.current_task.state.dynamic_priority:
                heapq.heappush(self.ready_queue, self.current_task)
                self.current_task = heapq.heappop(self.ready_queue)
                self.metrics.system_metrics["total_context_switches"] += 1

    def _calculate_quantum(self, task: Task) -> float:
        return QUANTUM_BASE * (2 if task.state.dynamic_priority < 8 else 1)

    def run(self, max_time=1000):
        while self.clock < max_time and (self.pending_tasks or self.ready_queue or self.current_task):
            while self.pending_tasks and self.pending_tasks[0][0] <= self.clock:
                _, task = heapq.heappop(self.pending_tasks)
                heapq.heappush(self.ready_queue, task)
                task.state.waiting_time = self.clock - task.config.arrival_time

            if not self.current_task and self.ready_queue:
                self.current_task = heapq.heappop(self.ready_queue)
                if self.current_task.state.start_time < 0:
                    self.current_task.state.start_time = self.clock
                    response_time = self.clock - self.current_task.config.arrival_time
                    self.metrics.task_metrics[self.current_task.config.task_id]["response_time"] = response_time

            if self.current_task:
                quantum = self._calculate_quantum(self.current_task)
                execute_time = min(quantum, self.current_task.state.remaining)

                self.current_task.state.remaining -= execute_time
                self.current_task.state.executed_time += execute_time
                self.total_busy_time += execute_time
                self.clock += execute_time

                for t in self.ready_queue:
                    t.state.waiting_time += execute_time

                if self.current_task.state.remaining <= 0:
                    self.current_task.state.end_time = self.clock
                    turnaround = self.clock - self.current_task.config.arrival_time
                    self.metrics.task_metrics[self.current_task.config.task_id].update({
                        "turnaround_time": turnaround,
                        "waiting_time": self.current_task.state.waiting_time,
                        "executed_time": self.current_task.state.executed_time,
                        "missed_deadline": self.clock > self.current_task.config.deadline
                    })
                    if self.clock > self.current_task.config.deadline:
                        self.metrics.system_metrics["missed_deadlines"] += 1
                    self.current_task = None
                    self.metrics.system_metrics["total_context_switches"] += 1
                else:
                    self._update_priorities()
                    heapq.heappush(self.ready_queue, self.current_task)
                    self.current_task = None

            self._handle_preemption()

        # Final metrics
        execution_times = [t["executed_time"] for t in self.metrics.task_metrics.values()]
        self.metrics.system_metrics.update({
            "cpu_utilization": self.total_busy_time / self.clock if self.clock > 0 else 0,
            "jains_fairness": self.metrics.calculate_jains_fairness(execution_times),
            "avg_waiting_time": np.mean([t["waiting_time"] for t in self.metrics.task_metrics.values()]),
            "avg_response_time": np.mean([t["response_time"] for t in self.metrics.task_metrics.values()])
        })
        return self.metrics


In [3]:
# --- Example Usage ---
    # task_id: int
    # arrival_time: float
    # expected_duration: float
    # deadline: float
    # is_io_bound: bool = False
    # is_foreground: bool = False  # OS decides boost based on this

if __name__ == "__main__":
    tasks = [
        # 🟢 I/O-bound + Foreground (should be boosted)
        TaskConfig(1, 0, 10, 20, is_io_bound=True, is_foreground=True),

        # 🔵 CPU-bound + Background (lowest priority)
        TaskConfig(2, 2, 12, 25, is_io_bound=False, is_foreground=False),

        # 🟠 I/O-bound + Background (might get priority for I/O)
        TaskConfig(3, 3, 8, 18, is_io_bound=True, is_foreground=False),

        # 🔴 CPU-bound + Foreground (needs boost for responsiveness)
        TaskConfig(4, 5, 15, 30, is_io_bound=False, is_foreground=True),

        # 🟣 Very short task, late arrival
        TaskConfig(5, 12, 2, 18, is_io_bound=True, is_foreground=True),

        # ⚫ Heavy long CPU-bound background task
        TaskConfig(6, 1, 25, 50, is_io_bound=False, is_foreground=False),

        # 🟡 Mid-range task, edge deadline
        TaskConfig(7, 4, 6, 10, is_io_bound=False, is_foreground=False),

        # 🧠 Foreground but late, I/O heavy
        TaskConfig(8, 14, 5, 20, is_io_bound=True, is_foreground=True),
    ]

    scheduler = Scheduler()
    for t in tasks:
        scheduler.add_task(Task(t))

    metrics = scheduler.run()

    print("\nSystem Metrics:")
    for k, v in metrics.system_metrics.items():
        print(f"{k:>20}: {v:.2f}")

    print("\nPer-Task Metrics:")
    for task_id, data in metrics.task_metrics.items():
        print(f"Task {task_id}:")
        for k, v in data.items():
            print(f"  {k:>16}: {v:.2f}")


System Metrics:
total_context_switches: 8.00
     cpu_utilization: 1.00
    missed_deadlines: 5.00
      jains_fairness: 0.70
    avg_waiting_time: 24.38
   avg_response_time: 17.50

Per-Task Metrics:
Task 1:
     response_time: 0.00
   turnaround_time: 10.00
      waiting_time: 0.00
     executed_time: 10.00
   missed_deadline: 0.00
Task 4:
     response_time: 5.00
   turnaround_time: 35.00
      waiting_time: 20.00
     executed_time: 15.00
   missed_deadline: 1.00
Task 5:
     response_time: 0.00
   turnaround_time: 2.00
      waiting_time: 0.00
     executed_time: 2.00
   missed_deadline: 0.00
Task 8:
     response_time: 0.00
   turnaround_time: 5.00
      waiting_time: 0.00
     executed_time: 5.00
   missed_deadline: 0.00
Task 3:
     response_time: 16.00
   turnaround_time: 24.00
      waiting_time: 16.00
     executed_time: 8.00
   missed_deadline: 1.00
Task 6:
     response_time: 39.00
   turnaround_time: 82.00
      waiting_time: 57.00
     executed_time: 25.00
   missed_dea

# CODE FOR GENERATING DATASET

In [1]:
import random
import json
from dataclasses import dataclass
from typing import List, Tuple

@dataclass
class TaskConfig:
    task_id: int
    arrival_time: float
    expected_duration: float
    deadline: float
    is_io_bound: bool = False
    is_foreground: bool = False

def generate_random_task(task_id: int, time: float) -> TaskConfig:
    return TaskConfig(
        task_id=task_id,
        arrival_time=time + random.uniform(0, 10),
        expected_duration=random.randint(1, 20),
        deadline=time + random.uniform(15, 60),
        is_io_bound=random.choice([True, False]),
        is_foreground=random.choice([True, False]),
    )

def encode_task(task: TaskConfig) -> List[float]:
    return [
        task.arrival_time,
        task.expected_duration,
        task.deadline,
        1.0 if task.is_io_bound else 0.0,
        1.0 if task.is_foreground else 0.0,
    ]

def heuristic_priority(tasks: List[TaskConfig], strategy="EDF") -> List[int]:
    """Assigns priority 0 (highest) to N-1 (lowest) using a strategy"""
    if strategy == "EDF":
        sorted_tasks = sorted(tasks, key=lambda t: t.deadline)
    elif strategy == "SJF":
        sorted_tasks = sorted(tasks, key=lambda t: t.expected_duration)
    elif strategy == "FCFS":
        sorted_tasks = sorted(tasks, key=lambda t: t.arrival_time)
    else:
        raise ValueError("Unknown strategy")
    
    priorities = [0] * len(tasks)
    for prio, task in enumerate(sorted_tasks):
        priorities[tasks.index(task)] = prio
    return priorities

def generate_dataset_row(tasks: List[TaskConfig], strategy="EDF") -> Tuple[List[List[float]], List[int]]:
    features = [encode_task(t) for t in tasks]
    priorities = heuristic_priority(tasks, strategy)
    return features, priorities

def generate_dataset(num_samples: int = 10000, max_tasks: int = 32, strategy="EDF") -> List[Tuple[List[List[float]], List[int]]]:
    dataset = []
    task_id = 0
    for _ in range(num_samples):
        num_tasks = random.randint(8, max_tasks)  # simulate dynamic task queues
        tasks = [generate_random_task(task_id + i, time=random.uniform(0, 50)) for i in range(num_tasks)]
        task_id += num_tasks
        features, priorities = generate_dataset_row(tasks, strategy)
        dataset.append((features, priorities))
    return dataset

def save_dataset(dataset, path="scheduling_dataset_test.json"):
    json_data = [{"input": row[0], "output": row[1]} for row in dataset]
    with open(path, "w") as f:
        json.dump(json_data, f, indent=2)

if __name__ == "__main__":
    data = generate_dataset(num_samples=20000, strategy="EDF")  # Or SJF / FCFS
    save_dataset(data)
    print("Dataset generated and saved.")


Dataset generated and saved.


In [3]:
!pip install -q datasets torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 69.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
bigframe

In [3]:
from datasets import load_dataset

# Load JSON dataset
dataset = load_dataset("json", data_files="/kaggle/working/scheduling_dataset.json")
# Check a sample
print(dataset["train"][0])


Generating train split: 0 examples [00:00, ? examples/s]

{'input': [[24.018280110374796, 8.0, 68.180259755831, 0.0, 1.0], [23.097318482773964, 19.0, 30.118919977293388, 1.0, 1.0], [31.799106498463665, 8.0, 49.18080591835147, 1.0, 1.0], [13.558593938013185, 17.0, 27.801293267526216, 1.0, 1.0], [47.73725693975541, 9.0, 80.83689483826609, 0.0, 0.0], [12.67714844614996, 12.0, 44.48097721021071, 1.0, 0.0], [24.15132515116637, 15.0, 62.3438841665182, 1.0, 1.0], [52.38751936890548, 6.0, 103.70121636992681, 1.0, 1.0], [31.378361391422725, 19.0, 46.98573336481184, 1.0, 1.0], [26.439380207439086, 1.0, 66.97529437390284, 0.0, 0.0], [49.93450343227428, 13.0, 94.82564304865323, 0.0, 0.0], [51.57289515727794, 19.0, 96.63570889882521, 1.0, 0.0], [28.03216152798508, 4.0, 67.66422592405237, 1.0, 0.0], [31.576429724999617, 3.0, 49.830348399117696, 0.0, 1.0], [24.307545962103735, 8.0, 56.861559592636674, 1.0, 1.0], [13.242402611505634, 9.0, 32.58061159487908, 0.0, 1.0], [6.863652488607057, 20.0, 34.86335200437502, 1.0, 0.0], [26.732970238374612, 2.0, 73.337200

In [4]:
from torch.utils.data import DataLoader
import torch

MAX_TASKS = 32
FEATURES_PER_TASK = 5

def collate_fn(batch):
    X, y = [], []
    for item in batch:
        task_set = item["task_set"]
        label = item["label"]

        # Each task is converted to a 5D feature vector
        task_features = [[
            t["arrival_time"], 
            t["duration"], 
            t["deadline"],
            float(t["is_io_bound"]),      # Ensure bools are cast to float
            float(t["is_foreground"])
        ] for t in task_set]

        # Pad with zeros to ensure all inputs are of shape [32 x 5]
        while len(task_features) < MAX_TASKS:
            task_features.append([0.0] * FEATURES_PER_TASK)

        X.append(task_features)
        y.append(label)  # This should be a list of 32 scores (one per task)

    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# Usage: assume `dataset["train"]` is a list of dicts with keys `task_set` and `label`
loader = DataLoader(
    dataset["train"], 
    batch_size=32, 
    collate_fn=collate_fn, 
    shuffle=True
)


In [7]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ------------ Dataset Loader ------------

class TaskPriorityDataset(Dataset):
    def __init__(self, json_file, max_tasks=32):
        with open(json_file, 'r') as f:
            self.data = json.load(f)

        self.max_tasks = max_tasks
        self.feature_dim = 5  # arrival_time, duration, deadline, is_io, is_fg

        self.samples = []
        for row in self.data:
            tasks = row['input']  # shape: (n_tasks, 5)
            order = row['output']  # list of priority indices

            # Create a priority score vector where higher score = higher priority
            # e.g., position 0 in `order` is most important → score = 1.0
            scores = [0.0] * len(tasks)
            for rank, task_idx in enumerate(order):
                scores[task_idx] = 1.0 - (rank / len(order))  # normalized

            # Pad both to max_tasks
            tasks = tasks[:max_tasks] + [[0.0] * self.feature_dim] * (max_tasks - len(tasks))
            scores = scores[:max_tasks] + [0.0] * (max_tasks - len(scores))

            self.samples.append((
                torch.tensor(tasks, dtype=torch.float32),
                torch.tensor(scores, dtype=torch.float32)
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ------------ Transformer Model ------------

class TaskPriorityTransformer(nn.Module):
    def __init__(self, input_dim=5, model_dim=64, num_heads=4, num_layers=3, max_tasks=32):
        super(TaskPriorityTransformer, self).__init__()
        self.max_tasks = max_tasks
        self.model_dim = model_dim

        self.input_proj = nn.Linear(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_tasks, model_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        x = self.transformer(x)
        scores = self.output_layer(x).squeeze(-1)
        return scores  # (batch_size, max_tasks)

# ------------ Training Code ------------

def train_model(json_path, epochs=50, batch_size=32, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = TaskPriorityDataset(json_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = TaskPriorityTransformer().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch_inputs, batch_targets in dataloader:
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

            optimizer.zero_grad()
            outputs = model(batch_inputs)
            loss = criterion(outputs, batch_targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    # Save model
    torch.save(model.state_dict(), "priority_transformer.pth")
    print("\nModel saved as 'priority_transformer.pth'")

# ------------ Run Training ------------

if __name__ == "__main__":
    # Ensure your dataset file path is correct
    train_model("/kaggle/working/scheduling_dataset.json")


Epoch 1/50, Loss: 0.0589
Epoch 2/50, Loss: 0.0525
Epoch 3/50, Loss: 0.0521
Epoch 4/50, Loss: 0.0520
Epoch 5/50, Loss: 0.0518
Epoch 6/50, Loss: 0.0518
Epoch 7/50, Loss: 0.0517
Epoch 8/50, Loss: 0.0517
Epoch 9/50, Loss: 0.0517
Epoch 10/50, Loss: 0.0516
Epoch 11/50, Loss: 0.0516
Epoch 12/50, Loss: 0.0516
Epoch 13/50, Loss: 0.0516
Epoch 14/50, Loss: 0.0515
Epoch 15/50, Loss: 0.0515
Epoch 16/50, Loss: 0.0514
Epoch 17/50, Loss: 0.0514
Epoch 18/50, Loss: 0.0512
Epoch 19/50, Loss: 0.0506
Epoch 20/50, Loss: 0.0499
Epoch 21/50, Loss: 0.0487
Epoch 22/50, Loss: 0.0476
Epoch 23/50, Loss: 0.0467
Epoch 24/50, Loss: 0.0459
Epoch 25/50, Loss: 0.0452
Epoch 26/50, Loss: 0.0447
Epoch 27/50, Loss: 0.0442
Epoch 28/50, Loss: 0.0437
Epoch 29/50, Loss: 0.0433
Epoch 30/50, Loss: 0.0429
Epoch 31/50, Loss: 0.0425
Epoch 32/50, Loss: 0.0423
Epoch 33/50, Loss: 0.0420
Epoch 34/50, Loss: 0.0417
Epoch 35/50, Loss: 0.0416
Epoch 36/50, Loss: 0.0413
Epoch 37/50, Loss: 0.0411
Epoch 38/50, Loss: 0.0409
Epoch 39/50, Loss: 0.

In [ ]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ------------ Dataset Loader ------------

class TaskPriorityDataset(Dataset):
    def __init__(self, json_file, max_tasks=32):
        with open(json_file, 'r') as f:
            self.data = json.load(f)

        self.max_tasks = max_tasks
        self.feature_dim = 5  # arrival_time, duration, deadline, is_io, is_fg

        self.samples = []
        for row in self.data:
            tasks = row['input']  # shape: (n_tasks, 5)
            order = row['output']  # list of priority indices

            # Create a priority score vector where higher score = higher priority
            # e.g., position 0 in `order` is most important → score = 1.0
            scores = [0.0] * len(tasks)
            for rank, task_idx in enumerate(order):
                scores[task_idx] = 1.0 - (rank / len(order))  # normalized

            # Pad both to max_tasks
            tasks = tasks[:max_tasks] + [[0.0] * self.feature_dim] * (max_tasks - len(tasks))
            scores = scores[:max_tasks] + [0.0] * (max_tasks - len(scores))

            self.samples.append((
                torch.tensor(tasks, dtype=torch.float32),
                torch.tensor(scores, dtype=torch.float32)
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ------------ Transformer Model ------------

class TaskPriorityTransformer(nn.Module):
    def __init__(self, input_dim=5, model_dim=64, num_heads=4, num_layers=3, max_tasks=32):
        super(TaskPriorityTransformer, self).__init__()
        self.max_tasks = max_tasks
        self.model_dim = model_dim

        self.input_proj = nn.Linear(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_tasks, model_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        x = self.transformer(x)
        scores = self.output_layer(x).squeeze(-1)
        return scores  # (batch_size, max_tasks)

# ------------ Training Code ------------

def train_model(json_path, epochs=100, batch_size=32, lr=5e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = TaskPriorityDataset(json_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = TaskPriorityTransformer().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch_inputs, batch_targets in dataloader:
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

            optimizer.zero_grad()
            outputs = model(batch_inputs)
            loss = criterion(outputs, batch_targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    # Save model
    torch.save(model.state_dict(), "priority_transformer1.pth")
    print("\nModel saved as 'priority_transformer.pth'")

# ------------ Run Training ------------

if __name__ == "__main__":
    # Ensure your dataset file path is correct
    train_model("/kaggle/working/scheduling_dataset.json")


Epoch 1/100, Loss: 0.0679
Epoch 2/100, Loss: 0.0521
Epoch 3/100, Loss: 0.0519
Epoch 4/100, Loss: 0.0518
Epoch 5/100, Loss: 0.0518
Epoch 6/100, Loss: 0.0518
Epoch 7/100, Loss: 0.0517
Epoch 8/100, Loss: 0.0517
Epoch 9/100, Loss: 0.0517
Epoch 10/100, Loss: 0.0517
Epoch 11/100, Loss: 0.0516
Epoch 12/100, Loss: 0.0515
Epoch 13/100, Loss: 0.0514
Epoch 14/100, Loss: 0.0512
Epoch 15/100, Loss: 0.0499
Epoch 16/100, Loss: 0.0483
Epoch 17/100, Loss: 0.0473
Epoch 18/100, Loss: 0.0466
Epoch 19/100, Loss: 0.0461
Epoch 20/100, Loss: 0.0456
Epoch 21/100, Loss: 0.0452
Epoch 22/100, Loss: 0.0446
Epoch 23/100, Loss: 0.0441
Epoch 24/100, Loss: 0.0436
Epoch 25/100, Loss: 0.0430
Epoch 26/100, Loss: 0.0425
Epoch 27/100, Loss: 0.0421
Epoch 28/100, Loss: 0.0418
Epoch 29/100, Loss: 0.0414
Epoch 30/100, Loss: 0.0411
Epoch 31/100, Loss: 0.0407
Epoch 32/100, Loss: 0.0404
Epoch 33/100, Loss: 0.0401
Epoch 34/100, Loss: 0.0398
Epoch 35/100, Loss: 0.0395
Epoch 36/100, Loss: 0.0392
Epoch 37/100, Loss: 0.0389
Epoch 38/1

# Training on a 50k dataset

In [4]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ------------ Dataset Loader ------------

class TaskPriorityDataset(Dataset):
    def __init__(self, json_file, max_tasks=32):
        with open(json_file, 'r') as f:
            self.data = json.load(f)

        self.max_tasks = max_tasks
        self.feature_dim = 5  # arrival_time, duration, deadline, is_io, is_fg

        self.samples = []
        for row in self.data:
            tasks = row['input']  # shape: (n_tasks, 5)
            order = row['output']  # list of priority indices

            # Create a priority score vector where higher score = higher priority
            # e.g., position 0 in `order` is most important → score = 1.0
            scores = [0.0] * len(tasks)
            for rank, task_idx in enumerate(order):
                scores[task_idx] = 1.0 - (rank / len(order))  # normalized

            # Pad both to max_tasks
            tasks = tasks[:max_tasks] + [[0.0] * self.feature_dim] * (max_tasks - len(tasks))
            scores = scores[:max_tasks] + [0.0] * (max_tasks - len(scores))

            self.samples.append((
                torch.tensor(tasks, dtype=torch.float32),
                torch.tensor(scores, dtype=torch.float32)
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ------------ Transformer Model ------------

class TaskPriorityTransformer(nn.Module):
    def __init__(self, input_dim=5, model_dim=64, num_heads=4, num_layers=3, max_tasks=32):
        super(TaskPriorityTransformer, self).__init__()
        self.max_tasks = max_tasks
        self.model_dim = model_dim

        self.input_proj = nn.Linear(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_tasks, model_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        x = self.transformer(x)
        scores = self.output_layer(x).squeeze(-1)
        return scores  # (batch_size, max_tasks)

# ------------ Training Code ------------

def train_model(json_path, epochs=100, batch_size=32, lr=5e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = TaskPriorityDataset(json_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = TaskPriorityTransformer().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch_inputs, batch_targets in dataloader:
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

            optimizer.zero_grad()
            outputs = model(batch_inputs)
            loss = criterion(outputs, batch_targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    # Save model
    torch.save(model.state_dict(), "priority_transformer1.pth")
    print("\nModel saved as 'priority_transformer.pth'")

# ------------ Run Training ------------

# if __name__ == "__main__":
#     # Ensure your dataset file path is correct
#     train_model("/kaggle/working/scheduling_dataset.json")


In [ ]:
json_file = '/kaggle/working/scheduling_dataset.json'
with open(json_file, 'r') as f:
    data = json.load(f)
    print("Sample JSON row:", data[0])  # 👈 Print one row to check actual keys

In [7]:
def infer(model, task_batch, device):
    model.eval()
    with torch.no_grad():
        task_batch = task_batch.to(device)
        outputs = model(task_batch)  # shape: (batch_size, max_tasks)
        preds = torch.argsort(outputs, dim=1, descending=True)  # ranked task indices
    return preds  # (batch_size, max_tasks)

In [5]:
from scipy.stats import kendalltau

def evaluate(model, dataset, device):
    model.eval()
    total_tau = 0.0
    count = 0

    with torch.no_grad():
        for inputs, targets in DataLoader(dataset, batch_size=1):  # one sample at a time
            inputs = inputs.to(device)
            outputs = model(inputs).squeeze(0)  # shape: (max_tasks)
            pred_order = torch.argsort(outputs, descending=True).cpu().tolist()

            # Reconstruct true order from scores
            true_order = torch.argsort(targets.squeeze(0), descending=True).cpu().tolist()

            tau, _ = kendalltau(pred_order, true_order)
            if tau is not None:
                total_tau += tau
                count += 1

    avg_tau = total_tau / count if count > 0 else 0.0
    print(f"\nAverage Kendall Tau correlation: {avg_tau:.4f}")
    return avg_tau

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TaskPriorityTransformer().to(device)
model.load_state_dict(torch.load("priority_transformer.pth"))

dataset = TaskPriorityDataset("/kaggle/working/scheduling_dataset.json")
evaluate(model, dataset, device)

/tmp/ipykernel_31/2518092886.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("priority_transformer.pth"))



Average Kendall Tau correlation: 0.4300


0.429981653225807

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TaskPriorityTransformer().to(device)
model.load_state_dict(torch.load("/kaggle/working/priority_transformer1.pth"))

dataset = TaskPriorityDataset("/kaggle/working/scheduling_dataset.json")
evaluate(model, dataset, device)

/tmp/ipykernel_32/3285705974.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/kaggle/working/priority_transformer1.pth"))



Average Kendall Tau correlation: 0.4387


0.4386897580645143

In [6]:
import torch
import json
from torch.utils.data import Dataset, DataLoader
from scipy.stats import kendalltau
import torch.nn as nn

# ---------------- Dataset ----------------
class TaskPriorityDataset(Dataset):
    def __init__(self, json_file, max_tasks=32):
        with open(json_file, 'r') as f:
            self.data = json.load(f)

        self.max_tasks = max_tasks
        self.feature_dim = 5

        self.samples = []
        for row in self.data:
            tasks = row['input']
            order = row['output']

            scores = [0.0] * len(tasks)
            for rank, task_idx in enumerate(order):
                scores[task_idx] = 1.0 - (rank / len(order))

            tasks = tasks[:max_tasks] + [[0.0] * self.feature_dim] * (max_tasks - len(tasks))
            scores = scores[:max_tasks] + [0.0] * (max_tasks - len(scores))

            self.samples.append((
                torch.tensor(tasks, dtype=torch.float32),
                torch.tensor(scores, dtype=torch.float32)
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# ---------------- Model ----------------
class TaskPriorityTransformer(nn.Module):
    def __init__(self, input_dim=5, model_dim=64, num_heads=4, num_layers=3, max_tasks=32):
        super(TaskPriorityTransformer, self).__init__()
        self.max_tasks = max_tasks
        self.model_dim = model_dim

        self.input_proj = nn.Linear(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_tasks, model_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        x = self.transformer(x)
        scores = self.output_layer(x).squeeze(-1)
        return scores

# ---------------- Evaluation ----------------
def evaluate(model, dataloader, device):
    model.eval()
    total_tau = 0.0
    valid_samples = 0

    with torch.no_grad():
        for inputs, true_scores in dataloader:
            inputs = inputs.to(device)
            true_scores = true_scores.to(device)

            predicted_scores = model(inputs)

            for i in range(inputs.size(0)):
                valid_len = (true_scores[i] > 0).sum().item()
                if valid_len < 2:
                    continue

                pred_order = torch.argsort(predicted_scores[i][:valid_len], descending=True)
                true_order = torch.argsort(true_scores[i][:valid_len], descending=True)

                tau, _ = kendalltau(pred_order.cpu().numpy(), true_order.cpu().numpy())
                if tau is not None:
                    total_tau += tau
                    valid_samples += 1

    avg_tau = total_tau / valid_samples if valid_samples > 0 else 0.0
    print(f"\n✅ Average Kendall Tau correlation: {avg_tau:.4f}")
    return avg_tau

# ---------------- Main ----------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    test_dataset = TaskPriorityDataset("scheduling_dataset_test.json")
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    model = TaskPriorityTransformer().to(device)
    model.load_state_dict(torch.load("/kaggle/input/priorityformer/pytorch/default/1/priority_transformer1.pth", map_location=device))

    evaluate(model, test_loader, device)


/tmp/ipykernel_30/3229045079.py:98: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/kaggle/input/priorityformer/pytorch/default/1/priority_t


✅ Average Kendall Tau correlation: 0.2685
